# MILP Toy Example

This notebook generates a synthetic AI data-center scheduling instance, builds the Gurobi MILP model, solves it, extracts the schedule, computes metrics, and plots hourly profiles.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.synthetic import generate_toy_dataset
from src.evaluation.metrics import compute_summary_metrics
from src.evaluation.plotting import plot_hourly_profiles
from src.evaluation.results import extract_hourly_results, extract_schedule
from src.milp.gurobi_model import build_milp_model
from src.milp.solve import solve_model

## 1. Generate Synthetic Data

Future real-data inputs can replace this section with files from `data/processed/`.

In [ ]:
jobs_df, hourly_df, clusters_df, config = generate_toy_dataset()

# TODO: Replace synthetic data with real input files from data/processed/
# jobs_df = pd.read_csv(PROJECT_ROOT / "data/processed/jobs.csv")
# hourly_df = pd.read_csv(PROJECT_ROOT / "data/processed/hourly_inputs.csv")
# clusters_df = pd.read_json(PROJECT_ROOT / "data/processed/clusters.json")

display(jobs_df)
display(hourly_df.head())
display(clusters_df)
config

## 2. Build And Solve The MILP

In [ ]:
model, variables = build_milp_model(jobs_df, hourly_df, clusters_df, config)
model.Params.OutputFlag = 1
solve_model(model)

print(f"Objective value: {model.ObjVal:,.2f}")

## 3. Extract The Selected Schedule

In [ ]:
schedule_df = extract_schedule(jobs_df, variables)
display(schedule_df[["job_id", "category", "assigned_cluster", "start_hour", "duration", "power"]])

## 4. Compute Hourly Profiles And Costs

In [ ]:
hourly_results = extract_hourly_results(hourly_df, variables)
metrics = compute_summary_metrics(hourly_results, config)

display(hourly_results)
for key, value in metrics.items():
    print(f"{key}: {value:,.2f}")

## 5. Plot Hourly Load And Energy-Source Profiles

In [ ]:
fig = plot_hourly_profiles(hourly_results)